<a href="https://colab.research.google.com/github/MateusCDev/ETL_NETBOX/blob/master/ETL_NetBox_v6_3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL NetBox v6.3.1 — selecione um POP depois do upload

**Correção do erro 403 no `/trace/`:** a v6.3.1 envia a autenticação do pynetbox também na consulta de rastreamento. Se o cabo já foi criado, use INTERLIGAR_COPIAS com as mesmas cópias/estado. Não é necessário migrar novamente os POPs. Em outra sessão, envie o último ZIP de relatórios para restaurar o lote.

Use o ZIP exportado manualmente do SharePoint. A lista de POPs é calculada a partir do arquivo enviado.

**APLICAR cria uma cópia em um Site novo. RETOMAR continua somente a própria cópia interrompida.** O inventário anterior é preservado. Somente EQUIPAMENTO/DIO cria dispositivos; DESTINO sozinho é referência externa.

As tags `migracao-planilha` e a tag única da execução identificam o Site e os objetos da cópia. Modelos reconhecidos recebem a grade completa, incluindo portas livres. Modelos genéricos preservam as portas descritas e têm capacidade máxima a confirmar.


## Uso

A v6.3.1 concilia nomes de equipamentos entre abas e corrige referências abreviadas do DIO quando há evidência recíproca única. Consulte CONCILIACAO_IDENTIDADES.xlsx. No ZIP recebido, Socorro tem uma OLT e 21 cabos locais resolvidos. A cópia antiga com OLT duplicada não é excluída.

1. Execute a etapa 1, envie o ZIP e escolha o POP na lista exibida. O painel sugere ligações entre POPs. Marque a caixa de um par somente depois de conferir as duas fibras com o CGR. Opcionalmente, envie revisão ou relatórios anteriores.
2. Uma única cópia ETL remota comprovada pode ser importada automaticamente para pares confirmados. Se houver várias, informe a escolhida por nome exato em Cópias já existentes. Deixe **SIMULAR** inicialmente. Informe a URL, ou use o secret `NETBOX_URL`; o token será solicitado em entrada oculta ou lido de `NETBOX_TOKEN`.
3. Execute a etapa 2. Confira o plano, o dicionário e as pendências no ZIP de relatórios baixado.
4. Para gravar, mude o modo para **APLICAR** no painel e execute a etapa 2. Para continuar a mesma cópia após falha, use **RETOMAR**.

A etapa 1 não consulta o NetBox. `Consultar/conectar NetBox` desmarcado permite simulação offline. Em Opções avançadas você pode escolher uma versão específica da planilha, a política das pendências e o limite de requisições por segundo.


In [1]:
# @title 1. Enviar arquivos e selecionar o POP
# Configuração principal. Mantenha SIMULAR na primeira execução.
POP_SELECIONADO = ""
MODO = "SIMULAR"
USAR_NETBOX = True
SITE_ID = 0  # Interno: não há Site de produção como destino.
# INDEPENDENTES cria o inventário local e os cabos resolvidos; relata as conexões pendentes.
POLITICA_PENDENCIAS = "INDEPENDENTES"
NETBOX_URL = ""
ARQUIVO_ENTRADA = ""
ARQUIVO_PREFERIDO = ""
ARQUIVO_MAPEAMENTO = ""
ARQUIVO_MANIFESTO = ""
REENVIAR_ARQUIVOS = True

# A partir daqui não é necessário editar. Só a biblioteca padrão é importada
# antes de instalar/verificar as dependências no Python desta execução.
import sys, os, json, subprocess, importlib, importlib.metadata
import base64, gzip, shutil, getpass, argparse
from pathlib import Path
from datetime import datetime, timezone

PAYLOAD_GZIP_BASE64 = "..."

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


def ensure_dependencies(include_widgets=False):
    requirements={
        'pynetbox':('pynetbox==7.9.0',(7,9,0),(7,9,1)),
        'openpyxl':('openpyxl>=3.1,<4',(3,1),(4,0)),
        'pandas':('pandas>=2.2,<4',(2,2),(4,0)),
        'requests':('requests>=2.31,<3',(2,31),(3,0)),
    }
    if include_widgets:requirements['ipywidgets']=('ipywidgets>=8,<9',(8,0),(9,0))
    missing=[]
    for name,(requirement,minimum,maximum) in requirements.items():
        try:
            version=importlib.metadata.version(name)
            numbers=tuple(int(p) for p in version.split('.') if p.isdigit())
            if not (minimum<=numbers<maximum):missing.append(requirement)
        except importlib.metadata.PackageNotFoundError:missing.append(requirement)
    if missing:
        print('Preparação: instalando dependências no Python desta execução...',flush=True)
        process=subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--quiet',*missing],capture_output=True,text=True)
        if process.returncode:
            raise RuntimeError('A instalação não foi concluída. Confira o acesso do Colab ao PyPI. Detalhe: '+(process.stderr or process.stdout)[-1600:])
    check=subprocess.run([sys.executable,'-c','import pynetbox,pandas,openpyxl,requests; print("Dependências importadas com sucesso.")'],capture_output=True,text=True)
    if check.returncode:raise RuntimeError('Falha ao importar as dependências em um Python novo: '+check.stderr[-1000:])
    print(check.stdout.strip(),flush=True)


def get_secret(name):
    if os.environ.get(name):return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:return None


def launch(settings=None):
    options=dict(state_dir='',rear_map_path='',registry_path='',copy_manifest_paths=[],existing_copy_names=[],input_paths=[],list_pops=False,review_path='',resume_path='',requests_per_second=2,retry_attempts=6,pop=POP_SELECIONADO,mode=MODO,use_netbox=USAR_NETBOX,site_id=SITE_ID,
                 policy=POLITICA_PENDENCIAS,url=NETBOX_URL,input_path=ARQUIVO_ENTRADA,
                 preferred=ARQUIVO_PREFERIDO,config_path=ARQUIVO_MAPEAMENTO,reupload=REENVIAR_ARQUIVOS,manifest_path=ARQUIVO_MANIFESTO)
    if settings:options.update(settings)
    if options['mode'] not in {'SIMULAR','APLICAR','RETOMAR','INTERLIGAR_COPIAS','PREVER_LIMPEZA'}:raise ValueError('Use SIMULAR, APLICAR, RETOMAR, INTERLIGAR_COPIAS ou PREVER_LIMPEZA')
    if options['pop'].strip().upper() in {'TODOS','ALL','*'}:raise ValueError('Escolha exatamente um POP')
    if options['mode']!='SIMULAR' and not options['use_netbox']:raise ValueError('APLICAR, RETOMAR e PREVER_LIMPEZA precisam de conexão ao NetBox. Use SIMULAR para análise offline.')
    colab=is_colab()
    base=Path(options.get('base_dir') or ('/content/netbox-etl-v6-3' if colab else 'work/netbox-etl-v6-3')).resolve()
    input_dir=base/'entradas';input_dir.mkdir(parents=True,exist_ok=True)
    code_dir=base/'code';package=code_dir/'netbox_etl';package.mkdir(parents=True,exist_ok=True)
    payload=json.loads(gzip.decompress(base64.b64decode(PAYLOAD_GZIP_BASE64)))
    for filename,source in payload['modules'].items():(package/filename).write_text(source,encoding='utf-8')
    ensure_dependencies()
    if options['mode']=='INTERLIGAR_COPIAS':
        paths=[]
    elif options['mode']=='PREVER_LIMPEZA':
        candidate=Path(options['manifest_path']) if options['manifest_path'] else None
        if candidate is not None and not candidate.is_file():candidate=input_dir/candidate.name
        if candidate is None or not candidate.is_file():
            if not colab:raise ValueError('Informe --manifesto para gerar a prévia de limpeza')
            from google.colab import files
            print('Envie manifesto_criacao.json da execução a revisar. Nenhuma exclusão será feita.',flush=True)
            uploaded=files.upload()
            if len(uploaded)!=1:raise ValueError('Envie exatamente um manifesto de criação')
            name,content=next(iter(uploaded.items()));candidate=input_dir/Path(name).name;candidate.write_bytes(content)
        options['manifest_path']=str(candidate.resolve());paths=[candidate]
    elif options['input_paths']:
        paths=[Path(p).expanduser().resolve() for p in options['input_paths']]
        if not all(p.is_file() for p in paths):raise ValueError('Arquivo de origem não encontrado; execute novamente a etapa de upload.')
    elif options['input_path']:

        paths=[Path(options['input_path']).expanduser().resolve()]
        if not paths[0].is_file():raise ValueError('ARQUIVO_ENTRADA não existe. Deixe vazio para fazer upload no Colab.')
    else:
        available=[p for p in input_dir.iterdir() if p.suffix.lower() in {'.xlsx','.xls','.zip'}]
        if options['reupload'] or not available:
            if not colab:raise ValueError('Informe --arquivo no terminal ou ARQUIVO_ENTRADA no Colab.')
            from google.colab import files
            print('Envie o ZIP exportado manualmente do SharePoint, ou o Excel do POP. Pode incluir um mapeamento JSON revisado.',flush=True)
            uploaded=files.upload()
            uploaded_sources=[]
            for name,content in uploaded.items():
                destination=input_dir/Path(name).name
                if destination.suffix.lower() in {'.xlsx','.xls','.zip','.json'}:destination.write_bytes(content)
                if destination.suffix.lower() in {'.xlsx','.xls','.zip'}:uploaded_sources.append(destination)
        paths=uploaded_sources if options['reupload'] or not available else sorted(p for p in input_dir.iterdir() if p.suffix.lower() in {'.xlsx','.xls','.zip'})
    if not paths and options['mode']!='INTERLIGAR_COPIAS':raise ValueError('Nenhum Excel ou ZIP foi enviado')
    sys.path.insert(0,str(code_dir))
    from netbox_etl.selection import pop_choices,select_pop
    if options['mode'] not in {'PREVER_LIMPEZA','INTERLIGAR_COPIAS'}:
        choices=pop_choices(paths)
        if not choices:raise ValueError('Nenhum POP encontrado no ZIP/Excel.')
        if options['list_pops']:
            for i,p in enumerate(choices,1):print(f'{i}. {p["label"]} | ID: {p["id"]}')
            return {'status':'POPS_LISTADOS','pops':choices}
        if not options['pop'].strip():
            for i,p in enumerate(choices,1):print(f'{i}. {p["label"]}')
            answer=input('Selecione o número do POP: ').strip()
            if not answer.isdigit() or not 1<=int(answer)<=len(choices):raise ValueError('Seleção de POP inválida')
            options['pop']=choices[int(answer)-1]['id']
        options['pop']=select_pop(options['pop'],choices)
    elif options['mode']=='INTERLIGAR_COPIAS':
        from netbox_etl.selection import workbook_site
        if not options['pop'].strip():raise ValueError('Escolha um POP para processar suas interligações')
        options['pop']=workbook_site(options['pop']+'.xlsx')
    elif not options['pop'].strip():
        options['pop']=json.loads(Path(options['manifest_path']).read_text())['pop']
    if options['mode']=='RETOMAR' and not options['resume_path']:raise ValueError('RETOMAR exige manifesto/ZIP de relatórios da interrupção.')

    config=payload['config']
    if options['config_path']:
        candidate=Path(options['config_path'])
        if not candidate.is_file():candidate=input_dir/Path(options['config_path']).name
        config=json.loads(candidate.read_text(encoding='utf-8-sig'))
    else:
        candidates=[]
        for path in input_dir.glob('*.json'):
            document=json.loads(path.read_text(encoding='utf-8-sig'))
            if isinstance(document,dict) and 'sites' in document:candidates.append((path,document))
        if len(candidates)>1:raise ValueError('Mais de um mapeamento JSON enviado. Preencha ARQUIVO_MAPEAMENTO com o nome desejado.')
        if candidates:config=candidates[0][1]
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    report_dir=base/'relatorios'/stamp;report_dir.mkdir(parents=True,exist_ok=True)
    config_path=report_dir/'mapeamento_entrada.json'
    config_path.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
    token='';url=''
    if options['use_netbox']:
        url=options['url'].strip() or get_secret('NETBOX_URL') or input('URL HTTPS do NetBox: ').strip()
        token=get_secret('NETBOX_TOKEN') or getpass.getpass('Token NetBox (entrada oculta): ')
    job=dict(paths=[str(p) for p in paths],pop=options['pop'],mode=options['mode'],
             use_netbox=options['use_netbox'],site_id=options['site_id'],policy=options['policy'],
             preferred=options['preferred'],netbox_url=url,config_path=str(config_path),output_dir=str(report_dir),strategy='ISOLATED_COPY',manifest_path=options['manifest_path'],review_path=options['review_path'],resume_path=options['resume_path'],requests_per_second=options['requests_per_second'],retry_attempts=options['retry_attempts'],state_dir=options['state_dir'] or str(base/'estado'),rear_map_path=options['rear_map_path'],registry_path=options['registry_path'],copy_manifest_paths=options['copy_manifest_paths'],existing_copy_names=options['existing_copy_names'])
    job_path=report_dir/'execucao.json';job_path.write_text(json.dumps(job,ensure_ascii=False,indent=2),encoding='utf-8')
    child_env=os.environ.copy()
    child_env['PYTHONPATH']=str(code_dir)+os.pathsep+child_env.get('PYTHONPATH','')
    if token:child_env['NETBOX_TOKEN']=token
    print(f'Executando {options["mode"]} para o POP {options["pop"]}. APLICAR cria uma nova cópia; RETOMAR continua somente a cópia do manifesto.',flush=True)
    process=None
    try:
        with (report_dir/'execucao.log').open('w',encoding='utf-8') as log:
            process=subprocess.Popen([sys.executable,'-u','-m','netbox_etl.worker',str(job_path)],env=child_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            for line in process.stdout:
                if token:line=line.replace(token,'[TOKEN OCULTO]')
                print(line,end='',flush=True);log.write(line);log.flush()
            process.wait()
    finally:
        # Parar a célula também para o worker antes de permitir uma retomada.
        if process is not None and process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        # O token nunca é salvo no JSON de execução, notebook ou relatório.
        child_env.pop('NETBOX_TOKEN',None);token=''
        saved_state=Path(options['state_dir'] or str(base/'estado'))
        for filename in ['estado_copias.json','ligacoes_rear.xlsx']:
            backup=saved_state/filename
            if backup.is_file():shutil.copyfile(backup,report_dir/filename)
        archive=Path(shutil.make_archive(str(report_dir),'zip',report_dir))
        print('\nRelatórios:',archive,flush=True)
        if colab:
            from google.colab import files
            files.download(str(archive))
    result_path=report_dir/'resultado.json'
    if not result_path.exists():
        print('O processo foi interrompido. Abra execucao.log; nenhum sucesso foi presumido.',flush=True)
        return {'status':'PROCESSO_INTERROMPIDO','report_dir':str(report_dir)}
    result=json.loads(result_path.read_text(encoding='utf-8'))
    result['report_dir']=str(report_dir)
    return result


def preparar_colab():
    base=Path('/content/netbox-etl-v6-3' if is_colab() else 'work/netbox-etl-v6-3').resolve()
    code=base/'code';package=code/'netbox_etl';package.mkdir(parents=True,exist_ok=True)
    payload=json.loads(gzip.decompress(base64.b64decode(PAYLOAD_GZIP_BASE64)))
    for filename,source in payload['modules'].items():(package/filename).write_text(source,encoding='utf-8')
    ensure_dependencies(include_widgets=True)
    sys.path.insert(0,str(code))
    # Recarrega somente os módulos leves do painel; execução ETL usa processo novo.
    for name in list(sys.modules):
        if name=='netbox_etl' or name.startswith('netbox_etl.'):sys.modules.pop(name,None)
    if is_colab():
        from google.colab import output
        output.enable_custom_widget_manager()
    from netbox_etl.colab_ui import prepare_panel
    return prepare_panel(base)

def executar_colab(panel):
    from netbox_etl.colab_ui import execute_panel
    return execute_panel(panel,launch)


def main():
    settings={}
    if not is_colab() and 'ipykernel' not in sys.modules:
        parser=argparse.ArgumentParser(description='ETL autônomo de um único POP. Padrão SIMULAR.')
        parser.add_argument('--arquivo',dest='input_path')
        parser.add_argument('--pop')
        parser.add_argument('--modo',dest='mode',choices=['SIMULAR','APLICAR','RETOMAR','INTERLIGAR_COPIAS','PREVER_LIMPEZA'])
        parser.add_argument('--ligacoes-rear',dest='rear_map_path')
        parser.add_argument('--registro',dest='registry_path')
        parser.add_argument('--importar-manifesto',dest='copy_manifest_paths',action='append')
        parser.add_argument('--copia-existente',dest='existing_copy_names',action='append')
        parser.add_argument('--state-dir')
        parser.add_argument('--listar-pops',dest='list_pops',action='store_true')
        parser.add_argument('--revisao',dest='review_path')
        parser.add_argument('--retomar-manifesto',dest='resume_path')
        parser.add_argument('--requisicoes-por-segundo',dest='requests_per_second',type=float)
        parser.add_argument('--sem-netbox',dest='use_netbox',action='store_false',default=None)
        parser.add_argument('--politica',dest='policy',choices=['BLOQUEAR','INDEPENDENTES'])
        parser.add_argument('--config',dest='config_path')
        parser.add_argument('--manifesto',dest='manifest_path')
        parser.add_argument('--base-dir')
        settings={k:v for k,v in vars(parser.parse_args()).items() if v is not None}
    try:
        return launch(settings)
    except Exception as error:
        print('Não foi possível iniciar:',str(error),flush=True)
        print('Confira os campos de configuração acima; não foi informado sucesso de migração.',flush=True)
        return {'status':'ERRO_DE_PREPARACAO'}


PAINEL_ETL = preparar_colab()


Preparação: instalando dependências no Python desta execução...
Dependências importadas com sucesso.
Envie o ZIP de origem. Opcional: planilha Ligacoes_rear, revisão e relatórios/manifestos das cópias existentes.


Saving Infra Voanet - Controle de Estações.zip to Infra Voanet - Controle de Estações.zip


Selecione o POP e execute a etapa 2. Nenhuma consulta à API foi iniciada.


In [2]:
# @title 2. Executar a seleção
if "PAINEL_ETL" not in globals():
    raise ValueError("Execute primeiro a etapa 1 e selecione um POP.")
RESULTADO_ETL = executar_colab(PAINEL_ETL)


Dependências importadas com sucesso.
Token NetBox (entrada oculta): ··········
Executando INTERLIGAR_COPIAS para o POP socorro. APLICAR cria uma nova cópia; RETOMAR continua somente a cópia do manifesto.
API: limite de 2.0 req/s; Retry-After e backoff ativos.
COPIA IMPORTADA POR NOME EXATO: POP-PARAMIRIM [MIGRACAO f62cd8db] [Site 128].
COPIA REMOTA AUSENTE: madre-de-deus. A ligação aguardará o registro/migração desse POP.
COPIA REMOTA UNICA IMPORTADA: POP-SOCORRO [MIGRACAO 270a95e1]
REAR AGUARDANDO: PTP SOCORRO - MADRE-DE-DEUS / madre-de-deus
REAR JA_EXISTENTE_PRESERVADO: socorro ↔ paramirim / cabo 2057.
INTERLIGACOES: {'AGUARDANDO_POP': 1, 'JA_EXISTENTE_PRESERVADO': 1}

RESULTADO: COPIA_REGISTRADA_AGUARDANDO_POP | completo: False

Relatórios: /content/netbox-etl-v6-3/relatorios/20261113T192258665003Z.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Corrigir o rastreamento da execução anterior

Para o erro 403 em `/trace/`, restaure o último ZIP de relatórios (ou mantenha o estado na mesma sessão), escolha Socorro e INTERLIGAR_COPIAS. O cabo é relido e preservado; o programa refaz o trace autenticado. Se a API ainda recusar a leitura, informa CABO_PRESENTE_TRACE_SEM_ACESSO e mantém a pendência. APLICAR cria outra cópia e não é o modo para essa conferência.

## Interligação automática entre cópias

Depois de APLICAR/RETOMAR, a v6.3.1 registra a cópia e processa os pareamentos confirmados do POP selecionado. Se a outra cópia não existir, registra AGUARDANDO_POP. Quando ela for migrada no mesmo lote, cria o cabo entre as portas informadas automaticamente após preflight da API.

**Paramirim já migrado:** se a escolha automática não for única, informe `POP-PARAMIRIM [MIGRACAO f62cd8db]` no campo Cópias já existentes, ou envie seu manifesto/relatórios. Migre apenas Socorro. Esta aplicação cria uma nova cópia corrigida; a cópia anterior duplicada permanece até sua revisão de limpeza. A importação confere tags e procedência; não recria Paramirim.

A planilha `Modelo_Ligacoes_Rear_v6_3_1.xlsx` é uma alternativa à confirmação pelo painel. O candidato Paramirim 1 ↔ Socorro 32 permanece com confirmado=NAO. Marque SIM somente quando o CGR confirmar as fibras. Uma linha não confirmada não cria cabo.

Se as duas cópias já estão prontas, use **INTERLIGAR_COPIAS**. Esse modo não cria Sites/dispositivos. Ocupação por outro cabo bloqueia a ligação, sem desconectar pontas. O código lê `/trace/` depois de gravar; se não confirmar o caminho, informa pendência e permite repetir sem duplicar.

## Tipos de ligação

DIO↔switch, switch↔DIO e switch↔switch são aceitos dentro e entre cópias. Na fila informe `tipo_a/equipamento_a/porta_a` e os mesmos campos de B, além dos POPs. Use tipo `interface`, `front` ou `rear` conforme a terminação física. O DIO mantém seu mapeamento interno Front↔Rear. As colunas antigas dio/rear continuam válidas.

O relatório separa cabo físico de caminho completo. Uma fibra sem os patches locais fica CABO_PRESENTE_CAMINHO_PARCIAL. Só INTERLIGADO significa que o `/trace/` confirmou as duas interfaces. Se o patch for acrescentado posteriormente, reexecute INTERLIGAR_COPIAS.

## Continuar em outra sessão

O registro e a fila ficam em `estado/`, e são incluídos no ZIP de relatórios. Envie o último ZIP de relatórios na etapa 1 para restaurar o lote. O programa volta a conferir a API; não usa apenas o snapshot antigo. Selecione o lote correto quando houver mais de uma cópia do mesmo POP.

APLICAR cria uma nova cópia local. Para repetir somente as interligações, use INTERLIGAR_COPIAS. Para continuar uma migração local interrompida, use RETOMAR com manifesto v6.3.1.

## Escopo e compatibilidade

Somente Sites com procedência de cópia ETL, no mesmo servidor. IPs/VLANs são preservados. A fase de interligação não envia PATCH para interfaces ou dispositivos. O único reparo permitido é preencher um mapa Front↔Rear vazio, 1:1, da cópia e necessário ao enlace confirmado; mapeamentos existentes e MPO são preservados.

A API 4.6.8 usa `rear_ports`. O ETL relê o Front Port para confirmar o vínculo. Modelos reconhecidos recebem portas livres; genéricos preservam as portas descritas e têm capacidade máxima a confirmar.

## API, relatórios e revisão

Padrão de 2 req/s, Retry-After, backoff e registro antes da criação. Consulte `interligacoes_rear.json`, `estado_copias.json` e `manifesto_interligacoes.json`. Confira também CONCILIACAO_IDENTIDADES.xlsx para os nomes reunidos e referências de portas adequadas. Sem fila de pareamentos, o log informa INTERLIGACOES NAO PROCESSADAS; uma sugestão não confirmada informa A_CONFIRMAR. As pendências locais continuam em `PENDENCIAS_PARA_CORRIGIR.xlsx`; colunas amarelas podem ser preenchidas e reenviadas com o mesmo ZIP.

Para revisar a limpeza dos enlaces traseiros, envie somente `manifesto_interligacoes.json` em PREVER_LIMPEZA. Para a cópia local, use `manifesto_criacao.json`. Nenhuma exclusão é automática.
